<a href="https://colab.research.google.com/github/kofisarf/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")


Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

question = "What is a noun"
answer = ask_llm(question)
print(answer)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question},
    ],
)
print(response.usage)


A noun is a word that refers to a person, place, thing, or idea. It is a fundamental part of language and is used to identify and describe objects, concepts, and individuals in the world around us.

Nouns can be classified into different categories, including:

1. **Proper nouns**: These are names of specific people, places, or organizations, such as "John", "London", or "Google".
2. **Common nouns**: These are general terms that can refer to any member of a category, such as "dog", "city", or "company".
3. **Concrete nouns**: These are nouns that refer to physical objects that can be seen or touched, such as "book", "chair", or "car".
4. **Abstract nouns**: These are nouns that refer to intangible concepts or ideas, such as "happiness", "freedom", or "love".
5. **Collective nouns**: These are nouns that refer to a group of people, animals, or things, such as "family", "herd", or "team".

Examples of nouns include:

* Person: "teacher", "student", "doctor"
* Place: "city", "park", "bea

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
The system role gives the model general instructions, while the user role gives the actual task. A token is a small piece of text that the model processes. The API uses tokens to measure how much text is being processed, so more text usually means more tokens and more cost.

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = [ask_llm(question, temperature=0.0) for _ in range(5)]
high_temp_answers = [ask_llm(question, temperature=1.2) for _ in range(5)]

print("TEMPERATURE = 0.0")
for i, ans in enumerate(low_temp_answers, start=1):
    print(i, ans)

print("TEMPERATURE = 1.2")
for i, ans in enumerate(high_temp_answers, start=1):
    print(i, ans)


TEMPERATURE = 0.0
1 Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", which could add a local touch to the product.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "su" means "save" or "keep", making this name straightforward and easy to understand.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security.
7. **Traders' Fund**: This name is simple and straightforward, emphasizing the product's purpose and target audience.

Choose the one that resonates with your targ

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the responses were more consistent. At temperature 1.2, the responses had more variation. For a loan decision-support system, I would use a low temperature because the answers should be consistent and not too random.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter_text}"

print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"])))
print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"])))

SUMMARY_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer.
Summarize loan application letters factually and neutrally.
Only use information stated in the letter. Do not invent or assume any detail.
Write exactly 3-4 sentences. Do not give an opinion on whether the loan should be approved."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

def summarize_letter(letter_text):
    return ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        temperature=0,
    )

v1_l002 = ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"]))
v2_l002 = summarize_letter(LETTERS["L002"])
v1_l006 = ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"]))
v2_l006 = summarize_letter(LETTERS["L006"])

print("V1 L002:", v1_l002)
print("V2 L002:", v2_l002)
print("V1 L006:", v1_l006)
print("V2 L006:", v2_l006)


Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season, and is willing to repay the loan as soon as possible, despite not having collateral at the moment.
Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and "trustworthy", and promises to repay the loan within a year when his businesses are successful.
V1 L002: Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He doesn't have collateral to offer but is promising to repay the loan as soon as possible.
V2 L002: Kwame Boateng, a 

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. Both were mostly correct, but V2 was better. V2 included important details that V1 missed, such as the fact that Kofi had not started any of his three businesses. V2 was also more careful by saying that his friends described him as business-minded instead of presenting it as a fact

2. A loan officer might only read the summary, so made-up information could lead to a wrong loan decision. This is called hallucination, where the AI gives information that is not actually in the original letter


### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance bank.
Extract these fields and return them as a single JSON object with EXACTLY these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
repayment_months (number or null).
If a field is not stated in the letter, use null. Do not guess.
Return ONLY the JSON object, no extra text, no markdown code fences.

Example letter:
"My name is John Mensah. I run a small hardware shop in Tema and I would like
a loan of GHS 5,000 to buy more stock. My brother will guarantee the loan."

Example output:
{
  "applicant_name": "John Mensah",
  "amount_ghs": 5000,
  "purpose": "buy more stock for hardware shop",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": null
}
"""

EXTRACT_PROMPT = "Extract the fields from this loan application:\n\n{letter_text}"

def extract_fields(letter_text, temperature=0):
    raw_reply = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=temperature,
    )
    cleaned_reply = raw_reply.strip()
    if cleaned_reply.startswith("```"):
        cleaned_reply = cleaned_reply.strip("`")
        cleaned_reply = cleaned_reply.replace("json", "", 1).strip()
    try:
        return json.loads(cleaned_reply)
    except json.JSONDecodeError:
        print("could not parse JSON:", raw_reply)
        return None

rows = []
for letter_id, letter_text in LETTERS.items():
    fields = extract_fields(letter_text)
    if fields is not None:
        fields["letter_id"] = letter_id
        rows.append(fields)

extracted_df = pd.DataFrame(rows)
extracted_df = extracted_df[["letter_id"] + [c for c in extracted_df.columns if c != "letter_id"]]
extracted_df


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,buy feed and 500 new layers for poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** The few-shot example should not come from the six letters because it could give the model the answer to one of the letters we are testing. Without the “use null, do not guess” rule, the model might make up information that is missing from a letter. Temperature 0 is best for extraction because we want the same correct answer each time, not random or creative answers.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

BRIEF_SYSTEM_PROMPT = """You are an assistant that prepares decision-support briefs for a
human loan officer at a microfinance institution. You do NOT make loan decisions.
Produce a brief with exactly these four sections:
1. Strengths - bullet points, grounded only in facts from the letter
2. Risks / Red flags - bullet points
3. Missing information the officer should request - bullet points
4. Suggested next step - ONE short suggestion (e.g. "invite for interview",
   "request documents", "flag for senior review").
Never write "approve" or "reject". The final decision is always made by a human."""

BRIEF_PROMPT = """Loan application letter:
{letter_text}

Extracted data:
{extracted_json}

Write the decision-support brief."""

def make_brief(letter_text, extracted_fields):
    extracted_json = json.dumps(extracted_fields, indent=2)
    return ask_llm(
        BRIEF_PROMPT.format(letter_text=letter_text, extracted_json=extracted_json),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0,
    )

briefs = {}
for letter_id, letter_text in LETTERS.items():
    row = extracted_df[extracted_df["letter_id"] == letter_id].iloc[0].to_dict()
    briefs[letter_id] = make_brief(letter_text, row)

for letter_id in ["L001", "L002", "L006"]:
    print(letter_id)
    print(briefs[letter_id])


L001
## Step 1: Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* She has a consistent monthly profit of GHS 900, demonstrating a viable business.
* Akosua has saved GHS 2,500 through the susu scheme over two years without missing a contribution, showing her ability to manage savings and commitments.
* She has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.

## Step 2: Risks / Red flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit, which might pose a risk if her business does not expand as planned.
* Expanding into frozen foods with a deep freezer could introduce new operational risks and costs that might affect her ability to repay the loan.
* The repayment plan of GHS 450 over 20 months is relatively high compared to her current monthly profit, which could strain her cash flow.

## Step 3: Missing information the

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Yes. For L003, the system found the right strengths, such as good experience, steady income, savings, and a guarantor. For L006, it found the main problems, such as businesses that had not started, no guarantor or collateral, and a weak repayment plan. Overall, the system’s judgment was correct.
2. The model should not make the final approve or reject decision because it does not have all the information a real loan officer would have, such as credit history and bank policies. A human should make the final decision because lending decisions can have serious effects on people.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

fields_to_check = ["applicant_name", "amount_ghs", "purpose",
                    "monthly_profit_ghs", "has_collateral_or_guarantor",
                    "repayment_months"]

def values_match(field, predicted, gold):
    if field == "applicant_name":
        return str(predicted).strip().lower() == str(gold).strip().lower()
    return predicted == gold

gold_letter_ids = list(GOLD.keys())
results = {field: [] for field in fields_to_check}

for letter_id in gold_letter_ids:
    predicted_row = extracted_df[extracted_df["letter_id"] == letter_id].iloc[0]
    gold_row = GOLD[letter_id]
    for field in fields_to_check:
        results[field].append(values_match(field, predicted_row[field], gold_row[field]))

accuracy_table = pd.DataFrame(results, index=gold_letter_ids).T
accuracy_table["accuracy"] = accuracy_table[gold_letter_ids].mean(axis=1)
accuracy_table


,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,False,False,False,0.000000
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

runs_temp0 = [extract_fields(LETTERS["L004"], temperature=0.0) for _ in range(5)]
runs_temp1 = [extract_fields(LETTERS["L004"], temperature=1.0) for _ in range(5)]

def summarize_runs(runs, label):
    valid = [r for r in runs if r is not None]
    unique = len(set(json.dumps(r, sort_keys=True) for r in valid))
    print(label, "valid:", len(valid), "/", len(runs), "unique:", unique)

summarize_runs(runs_temp0, "temperature=0.0")
summarize_runs(runs_temp1, "temperature=1.0")


temperature=0.0 valid: 5 / 5 unique: 1
temperature=1.0 valid: 5 / 5 unique: 1


### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

test1_question = "What is the applicant's credit score?"
test1_answer = ask_llm(
    f"Here is a loan application letter:\n\n{LETTERS['L002']}\n\nQuestion: {test1_question}",
    system_prompt=SUMMARY_SYSTEM_PROMPT,
    temperature=0,
)
print(test1_answer)

weather_report = """Today's weather in Accra: sunny with a high of 31C, humid,
light winds from the southwest, chance of rain in the evening."""

test2_result = extract_fields(weather_report)
print(test2_result)


The loan application letter does not mention the applicant's credit score. The applicant, Kwame Boateng, is requesting GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to pick up after the festive season. The letter does not provide any information about his credit history or score.
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. The model got 100% for applicant name, loan amount, collateral/guarantor, and repayment months. It got 66.7% for monthly profit and 0% for purpose. Purpose was difficult because the model used different words even when the meaning was correct. For L006, it also guessed the monthly profit instead of saying null.

2. At both temperature 0 and 1, all five answers for L004 were valid and exactly the same. Even so, I would use temperature 0 because it is safer and more consistent for this type of task.

3. The model passed both tests, but it still guessed the monthly profit for L006. To reduce this problem, I would check that every number the model gives actually appears in the original letter.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. People who do not write English well could be treated unfairly, even if they have a good business. The model judges the writing instead of the actual business.

2. Loan applications contain private information. I would check how the API provider protects the data, whether they use it for training, and if it follows Ghana’s data protection rules.

3.  I would use human review and monitoring. A loan officer should check the AI’s work before making a decision. The bank should also regularly check the AI’s results to make sure it is not treating certain groups unfairly.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1. Both work in a similar way: change something, test it, check the result, and improve it. In Lab 3, I changed numbers like the learning rate. Here, I changed the wording of the prompt. The main difference is that prompts use language, so small wording changes can have a big effect

2.  would not fully trust the system on its own because it guessed the monthly profit for L006 even though it was told not to guess

3. My test used 343 tokens. With about 3 calls per application, that would be around 1,000 tokens per application. For 1,000 applications, that would be about 1 million tokens per month, so the bank would likely need a paid plan

4. I could not train a model like this myself because it would need a lot of data and computing power. Using an API is easier because the model already understands language. Training your own model would make more sense for a specific task or if the bank needed everything to run offline

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.